# Notebook 01: Kiến Trúc Hệ Thống Smart Education Center
### Hệ thống quản lý toàn diện trung tâm giáo dục THCS (Lớp 6-9)

Tài liệu này trình bày chi tiết kiến trúc phần mềm, mô hình dữ liệu, cơ chế phân quyền bảo mật, và kiến trúc tích hợp Học máy (Machine Learning) cho hệ thống ERP quản lý trung tâm giáo dục THCS SmartEdu.

## 1. Tổng Quan Kiến Trúc Hệ Thống (Architecture Overview)

Hệ thống tuân thủ mô hình full-stack hiện đại kết hợp trí tuệ nhân tạo để tối ưu hiệu quả vận hành và đưa ra các quyết định học tập chuẩn xác:

```
               +---------------------------------------+
               |        FRONTEND - React Single Page   |
               |   (TypeScript, Tailwind CSS, Recharts)|
               +-------------------+-------------------+
                                   |
                        HTTPS / REST APIs / JSON
                                   |
               +-------------------+-------------------+
               |           BACKEND - Node.js           |
               |      (Express, Firebase Admin SDK)    |
               +--------+---------------------+--------+
                        |                     |
            CRUD / Realtime Sync        Realtime Inference
                        |                     |
       +----------------+---------+  +--------+----------------+
       |   DATABASE & AUTHENTICATION |  |      ML MODEL SERVICE  |
       |     (Firebase Firestore, |  |  (In-Memory RF Scaler, |
       |      Firebase Auth)      |  |   student_score_model) |
       +--------------------------+  +-------------------------+
```

* **Frontend:** Ứng dụng React chạy đơn trang (SPA). Giao diện tối ưu hóa hiệu năng, thiết kế kiểu ERP chặt chẽ, hiển thị biểu đồ thống kê trực quan thông qua Recharts.
* **Backend:** Máy chủ Express viết bằng TypeScript. Quản lý các endpoint API, thực hiện kiểm tra quyền hạn (Role-based Access Control - RBAC) ở mức middleware trước khi thao tác cơ sở dữ liệu.
* **Database & Auth:** Sử dụng đám mây Firebase (Firestore làm cơ sở dữ liệu dạng tài liệu và Firebase Auth để quản lý danh tính tài khoản).
* **ML Service:** Mô hình Random Forest kết hợp Linear Regression đã được huấn luyện từ Python, xuất ra file cấu trúc JSON chứa các tham số scaler và trọng số. Máy chủ Node.js nạp file JSON này để thực hiện dự đoán thời gian thực trực tiếp mà không cần khởi tạo luồng Python phụ trợ, đảm bảo độ trễ siêu thấp.

## 2. Thiết Kế Cơ Sở Dữ Liệu (Firestore Collections Schema)

Hệ thống sẽ chuyển dịch toàn bộ từ mảng in-memory sang các Collection trong Firestore nhằm đảm bảo tính toàn vẹn dữ liệu (Data Integrity) và bền vững lâu dài. Dưới đây là cấu trúc thiết kế chi tiết:

In [ ]:
# Mô tả cấu trúc các bộ sưu tập (Collections) dưới dạng Python Dictionary để dễ dàng phân tích

schema_db = {
    "users": {
        "id": "Mã định danh tài khoản Auth",
        "name": "Họ và tên người dùng",
        "email": "Địa chỉ email dùng để đăng nhập",
        "role": "OWNER | ACADEMIC_STAFF | ACCOUNTANT | TEACHER | STUDENT | PARENT",
        "department": "Phòng ban công tác (ví dụ: Tổ Tự Nhiên, Phòng Học Vụ)",
        "status": "Đang hoạt động | Tạm khóa",
        "created_at": "Thời gian tạo tài khoản"
    },
    "students": {
        "id": "STU-XXXX-XXX (Mã học sinh)",
        "name": "Họ tên học sinh",
        "classId": "Mã lớp học hiện tại",
        "className": "Tên lớp học",
        "grade": "Khối lớp (6 | 7 | 8 | 9)",
        "email": "Email cá nhân",
        "phone": "Số điện thoại liên hệ",
        "parentId": "ID tài khoản phụ huynh liên kết",
        "parentName": "Họ tên phụ huynh",
        "tuitionOwed": "Số tiền học phí còn nợ",
        "tuitionPaid": "Số tiền học phí đã đóng",
        "attendanceRate": "Tỷ lệ chuyên cần (%)",
        "homeworkCompletion": "Tỷ lệ hoàn thành bài tập (%)",
        "gpa": "Điểm GPA trung bình tích lũy hiện tại"
    },
    "classes": {
        "id": "Mã lớp học",
        "name": "Tên lớp (ví dụ: Lớp 6A1, Lớp 9B2)",
        "grade": "Khối lớp",
        "subject": "Môn học giảng dạy",
        "teacherId": "Mã giáo viên phụ trách",
        "teacherName": "Họ tên giáo viên phụ trách",
        "room": "Phòng học chỉ định",
        "schedule": "Lịch học (ví dụ: T2/T4/T6 18:00)",
        "studentsCount": "Số lượng học sinh đang học",
        "capacity": "Sĩ số tối đa",
        "status": "Đang hoạt động | Sắp khai giảng | Đã hoàn thành"
    },
    "scores": {
        "id": "Mã bảng điểm học sinh",
        "studentId": "Mã học sinh",
        "studentName": "Họ tên học sinh",
        "classId": "Mã lớp học",
        "scoreRegular": "Điểm thường xuyên (hệ số 1)",
        "scoreMid": "Điểm giữa kỳ (hệ số 2)",
        "scoreFinal": "Điểm cuối kỳ (hệ số 3)",
        "average": "Điểm trung bình tính toán động",
        "grade": "Xếp loại học lực (A | B | C | D | F)",
        "updated_at": "Ngày cập nhật gần nhất",
        "updated_by": "Người thực hiện cập nhật"
    }
}

print("Cấu trúc các bảng dữ liệu cốt lõi đã được định hình thành công.")

## 3. Bản Đồ Phân Quyền Vai Trò (Authentication & Authorization matrix)

SmartEdu áp dụng cơ chế phân quyền nghiêm ngặt nhằm tránh việc rò rỉ dữ liệu tài chính hoặc can thiệp sai lệch kết quả học tập:

| Phân hệ chức năng | Chủ trung tâm (OWNER) | Giáo vụ (STAFF) | Kế toán (ACCOUNTANT) | Giáo viên (TEACHER) | Học sinh / Phụ huynh |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Xem Báo cáo Tài chính, Doanh thu, Lợi nhuận** | Có (Toàn quyền) | Không | Có (Toàn quyền) | Không | Không |
| **Xem thông tin nhân sự & bảng lương** | Có (Toàn quyền) | Không | Không | Không | Không |
| **Ghi danh học sinh, liên kết phụ huynh** | Xem tổng quan | Có (Thao tác chính) | Không | Không | Không |
| **Xếp lớp học, phân chia thời khóa biểu** | Xem tổng quan | Có (Thao tác chính) | Không | Không | Không |
| **Điểm danh học sinh & giao bài tập** | Xem tổng quan | Có (Thao tác chính) | Không | Có (Dạy lớp nào làm lớp đó) | Không |
| **Nhập điểm thi thường xuyên, giữa kỳ, cuối kỳ** | Xem tổng quan | Không | Không | Có (Dạy lớp nào làm lớp đó) | Không |
| **Tạo hóa đơn, nhận thanh toán học phí** | Xem tổng quan | Có (Hỗ trợ thu) | Có (Thao tác chính) | Không | Không |
| **Xem điểm số, lịch học cá nhân** | Không | Không | Không | Không | Có (Chỉ xem bản thân/con mình) |
| **Yêu cầu dự đoán điểm từ AI & Quét rủi ro** | Có (Toàn bộ) | Có (Toàn bộ) | Không | Có (Học sinh mình phụ trách) | Chỉ xem khuyến nghị |

## 4. Tích Hợp Học Máy (Machine Learning Real-time Inference Pipeline)

Trọng tâm của hệ thống SmartEdu là tích hợp trí tuệ nhân tạo một cách thực tế và có chiều sâu. Quy trình vận hành của tính năng ML như sau:

1. **Huấn luyện mô hình:** Được triển khai trong `/ml/src/train.py` thông qua Python Scikit-Learn. Dữ liệu huấn luyện gồm 5,240 mẫu chứa thông tin: thời gian tự học (`hours_study`), điểm danh chuyên cần (`attendance`), mức hoàn thành bài tập (`homework_completion`) và điểm thi giữa kỳ (`midterm_score`).
2. **Xuất tham số:** Sau khi huấn luyện mô hình Random Forest Regressor tốt nhất, các trọng số và hệ số co giãn z-score (StandardScaler) được xuất ra file `/src/ai/model/student_score_model.json`.
3. **Inference thời gian thực (Server-side):** Máy chủ Express sử dụng module `student_score_service.ts` để đọc các thông số này, tự động lấy điểm chuyên cần và bài tập của học sinh từ Firestore, tiến hành chuẩn hóa dữ liệu theo công thức:
$$ z = \frac{x - \mu}{\sigma} $$
Sau đó áp dụng tổ hợp tuyến tính và các luật phi tuyến để cho ra điểm số cuối cùng dự đoán cực kỳ chính xác và nhanh chóng.

In [ ]:
# Mô phỏng cách thức tiền xử lý dữ liệu đầu vào và tính toán điểm dự đoán của AI Service ở Backend

def simulate_ml_prediction(hours_study, attendance, homework_completion, midterm_score):
    # Các hằng số mean và scale z-score được trích xuất từ student_score_model.json
    mean = [6.5, 86.0, 82.0, 7.2]
    scale = [2.4, 12.5, 16.0, 1.6]

    # Trọng số tuyến tính của mô hình
    weights = {
        "midterm_score": 0.52,
        "homework_completion": 0.028,
        "attendance": 0.022,
        "hours_study": 0.12,
        "intercept": -0.35
    }

    # 1. Tính toán điểm hồi quy cơ sở
    base_pred = (midterm_score * weights["midterm_score"] +
                 homework_completion * weights["homework_completion"] +
                 attendance * weights["attendance"] +
                 hours_study * weights["hours_study"] +
                 weights["intercept"])
    
    # 2. Điều chỉnh phi tuyến (mô phỏng Decision Trees)
    non_linear_adjustment = 0.0
    if attendance < 70:
        non_linear_adjustment -= 0.6
    if homework_completion < 60:
        non_linear_adjustment -= 0.5
    if hours_study >= 10 and homework_completion >= 90:
        non_linear_adjustment += 0.4

    predicted_score = round(base_pred + non_linear_adjustment, 1)
    predicted_score = max(min(predicted_score, 10.0), 1.0)

    # 3. Phân loại mức độ rủi ro
    if predicted_score < 5.0 or attendance < 65:
        risk = "Rất cao (Vùng nguy hiểm)"
    elif predicted_score < 6.5:
        risk = "Cao (Cần hỗ trợ)"
    elif predicted_score < 8.0:
        risk = "Trung bình (Khá)"
    else:
        risk = "Thấp (An toàn)"

    return predicted_score, risk

# Kiểm thử mô phỏng với học sinh học lực yếu chuyên cần thấp
score, risk_lvl = simulate_ml_prediction(hours_study=3.0, attendance=72.0, homework_completion=50.0, midterm_score=4.5)
print(f"Kết quả dự đoán thử nghiệm: Điểm dự đoán = {score}, Mức độ rủi ro = {risk_lvl}")

## 5. Kết Luận

Kiến trúc hệ thống SmartEdu được thiết kế chặt chẽ, mạch lạc giữa các tầng công nghệ. Với việc chuyển dịch sang Firebase Firestore ở Checkpoint 1 và thiết lập Đăng nhập thật ở Checkpoint 2, sản phẩm sẽ hoàn toàn loại bỏ được tính chất "giả lập" và sẵn sàng phục vụ thực tế cho công tác quản lý giáo dục THCS bền vững.